In [1]:
%cd /cephyr/users/felixnie/Alvis/NeuroCBIR/

/cephyr/users/felixnie/Alvis/NeuroCBIR


**COMPUTATIONAL TIME: BRAIN STRUCTURES**

In [2]:
from memory_profiler import memory_usage

# Function to just return nothing, for baseline memory snapshot
def baseline_memory():
    return None

# Peak memory at the very beginning
baseline_peak = max(memory_usage(baseline_memory, interval=0.01, max_iterations=10))
print(f"Baseline memory before loading any model: {baseline_peak:.2f} MB")


Baseline memory before loading any model: 71.11 MB


In [3]:
from monai.networks.nets.autoencoderkl import Encoder
from monai.networks.nets.autoencoderkl import AutoencoderKL
from model.contrastive_model import ContrastiveModel
import matplotlib.pyplot as plt
import torch
import os
import psutil
import time
import gc
from tqdm import tqdm

gc.collect()  # Clean up any unreachable objects
%load_ext memory_profiler


# # Limit PyTorch to 1 thread
# torch.set_num_threads(1)

# # Optional: also restrict OMP and MKL
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"

# AUTOENCODER
device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
config_vae = {
    "random_state": 1234,
    "data_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/",
    "save_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/region_brain/",
    "metadata_file": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/combined_metadata.csv", 
    "labels_path": "data/labels.csv",
    "bb_path": "data/bounding_boxes.csv",
    "ckpt_path": "/cephyr/users/felixnie/Alvis/logs/20250725_110120/checkpoint-epoch-12.pth",
    "use_old_state_dict": False,
    "n_structs": -1,
    "vae_params": {
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 1,
        "latent_channels": 8,
        "channels": [64, 128, 128, 128],
        "num_res_blocks": 2,
        "norm_num_groups": 32,
        "norm_eps": 1e-6,
        "attention_levels": [False, False, False, False],
        "with_decoder_nonlocal_attn": False,
        "with_encoder_nonlocal_attn": False,
    },
    "batch_size": 64,
}

# CL PROJECTOR
config_cl = {
    "data_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/",
    "output_dir": "data/results/region_brain/eval_cl16/",
    "dataset_index_file_name": "region_brain/dataset_index.csv",
    "metadata_file_name": "combined_metadata.csv",
    "labels_path": "data/labels.csv",
    "bb_path": "data/bounding_boxes.csv",
    "proj_params": {
        "input_shape": [8, 8, 8, 8],
        "projector_dims": [128],
        "final_dim": 16
    },
    "encoder_params": {
        "spatial_dims": 3,
        "in_channels": 8,
        "channels": [64, 128, 256],
        "out_channels": 256,
        "num_res_blocks": [2, 2, 2],
        "norm_num_groups": 8,
        "norm_eps": 1e-5,
        "attention_levels": [False, False, False],
        "with_nonlocal_attn": False,
        "include_fc": False,
    },
}
config_cl.update({
    "resume_path": "/cephyr/users/felixnie/Alvis/logs/20250804_001927/checkpoint-epoch-495.pth",
    "n_structs": -1,
    "batch_size": 256
})



In [5]:
# Peak memory at the very beginning
baseline_peak = max(memory_usage(baseline_memory, interval=0.01, max_iterations=10))
print(f"Baseline memory before loading any model: {baseline_peak:.2f} MB")

Baseline memory before loading any model: 749.75 MB


In [6]:
import torch
import time
import gc
from memory_profiler import memory_usage

# ----------------------------
# Function to measure memory usage during a callable
# ----------------------------
def measure_memory(func, *args, **kwargs):
    """
    Runs func(*args, **kwargs) and returns:
      - result: function output
      - peak_mem: peak memory during execution in MB
    """
    peak_mem = memory_usage((func, args, kwargs), interval=0.01, max_iterations=1)
    return peak_mem, func(*args, **kwargs)

# ----------------------------
# VAE loading
# ----------------------------
def load_vae_model(config, device, ckpt_path=None, use_old_state_dict=False):
    vae_params = config["vae_params"]
    autoencoder = AutoencoderKL(**vae_params).to(device)

    if ckpt_path:
        checkpoint = torch.load(ckpt_path, map_location=device)
        ckpt_key = config.get("ckpt_key", "autoencoder_state_dict")
        if use_old_state_dict:
            autoencoder.load_old_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_old_state_dict().")
        else:
            autoencoder.load_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_state_dict().")
    return autoencoder

# ----------------------------
# Measure memory for VAE load
# ----------------------------
gc.collect()
time_before = time.time()
mem_peak_vae, autoencoder = measure_memory(load_vae_model,
                                           config=config_vae,
                                           device=device,
                                           ckpt_path=config_vae.get("ckpt_path"),
                                           use_old_state_dict=config_vae.get("use_old_state_dict"))
autoencoder.eval()
time_after = time.time()
print(f"VAE Load Time: {time_after - time_before:.2f} s")
print(f"Peak memory during VAE load: {max(mem_peak_vae):.2f} MB")

# ----------------------------
# Contrastive Learning (CL) projector
# ----------------------------
def create_encoder(config, device):
    encoder_params = config["encoder_params"]
    return Encoder(**encoder_params).to(device)

def build_cl_model(config, device):
    encoder = create_encoder(config, device)
    model = ContrastiveModel(
        encoder=encoder,
        input_shape=config["proj_params"]["input_shape"],
        projector_dims=config["proj_params"]["projector_dims"],
        final_dim=config["proj_params"]["final_dim"],
        device=device
    ).to(device)
    checkpoint = torch.load(config["resume_path"], map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    return model

gc.collect()
time_before_cl = time.time()
mem_peak_cl, model = measure_memory(build_cl_model, config_cl, device)
model.eval()
time_after_cl = time.time()

print(f"CL Model Load Time: {time_after_cl - time_before_cl:.2f} s")
print(f"Peak memory during CL load: {max(mem_peak_cl):.2f} MB")


Loaded weights using load_state_dict().
Loaded weights using load_state_dict().
VAE Load Time: 0.62 s
Peak memory during VAE load: 998.50 MB
CL Model Load Time: 0.47 s
Peak memory during CL load: 999.69 MB


In [7]:
from preprocessing.load_dataset import SubCorBatDataset
import pandas as pd
import os
import numpy as np

### Input data
# Path to dataset
DATA_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/"
load_ds_path = DATA_PATH + "batched_adni/"
# Files to load/save extension
extension = ".npz"
# Pretrained weights for the VAE
ckpt_vae_path = ""#"./data/pretrained_models/autoencoder_puglisi.pth"
# Pretrained weights for the Discriminator
ckpt_dis_path = ""#"./data/pretrained_models/ckpt_dis_.pth"
# Preparing image for using as input of the VAE
target_shape = [1, 160, 224, 160] # Desired shape: [1, 160, 224, 160]

# Load metadata
index_ds = pd.read_csv(os.path.join(DATA_PATH,"original/dataset_index.csv"))
clinical_ds = pd.read_csv(os.path.join(DATA_PATH,"combined_metadata.csv"))
metadata = pd.merge(index_ds, clinical_ds, on="GUID", how="inner") # Merge on the 'GUID' column
print(f"METADATA: Original rows: {len(metadata)}")

# First, ensure empty strings are treated as NaN
metadata['subject'].replace('', pd.NA, inplace=True)

# Then drop rows where subject is NaN
metadata = metadata.dropna(subset=['subject'])

# Optional: reset index
metadata = metadata.reset_index(drop=True)
print(f"METADATA: Remaining rows: {len(metadata)}") # Check result

# Load labels

# Training configuration
batch_files = sorted(metadata["batch_file"].unique())

# Load labels and bounding boxes
labels_df = pd.read_csv("data/labels.csv")
bb_df = pd.read_csv("data/bounding_boxes.csv")
labels_bb_df = pd.merge(labels_df, bb_df, on="LabelName", how="inner") # Merge on the 'GUID' column

batch_files = batch_files[-4:-3] # To only load ABLI
print(batch_files)

from preprocessing.load_dataset import SingleStructDataset
dataset = SingleStructDataset(metadata, batch_files, labels_bb_df, target_struct_name="Left-Hippocampus")

guids = np.array(dataset.guids)

sample = dataset[0].copy()
struct = sample["image"].unsqueeze(0).type(torch.float32).to(device)

del dataset
gc.collect()

METADATA: Original rows: 26685
METADATA: Remaining rows: 26685
['/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/batched_aibl/batch_0002.npz']


0

In [8]:
times_total = []
times_encode = []
times_project = []

n_iter = 10

for _ in tqdm(range(n_iter)):
    

    tic = time.time()
    with torch.no_grad():
        z_mu_batch, _ = autoencoder.encode(struct)
    toc = time.time()
    with torch.no_grad():
        proj_embs = model(z_mu_batch).cpu().numpy()
    tuc = time.time()

    times_total.append(tuc - tic)
    times_encode.append(toc - tic)
    times_project.append(tuc - toc)


100%|██████████| 10/10 [00:35<00:00,  3.59s/it]


In [9]:
from memory_profiler import memory_usage

mem_total = []
mem_encode = []
mem_project = []



for _ in tqdm(range(n_iter)):
    # Encode memory
    def encode_step():
        with torch.no_grad():
            z_mu_batch, _ = autoencoder.encode(struct)
        return z_mu_batch

    mem_encode_peak = max(memory_usage(encode_step, interval=0.01))
    z_mu_batch = encode_step()  # actually run

    # Project memory
    def project_step():
        with torch.no_grad():
            proj_embs = model(z_mu_batch).cpu().numpy()
        return proj_embs

    mem_project_peak = max(memory_usage(project_step, interval=0.01))
    proj_embs = project_step()

    # Total memory for this iteration
    mem_total_peak = max(mem_encode_peak, mem_project_peak)

    mem_encode.append(mem_encode_peak)
    mem_project.append(mem_project_peak)
    mem_total.append(mem_total_peak)


100%|██████████| 10/10 [01:13<00:00,  7.31s/it]


In [10]:

# Convert to numpy
times_total = np.array(times_total)
times_encode = np.array(times_encode)
times_project = np.array(times_project)

mem_total = np.array(mem_total)
mem_encode = np.array(mem_encode)
mem_project = np.array(mem_project)

# Print results
print(f"\n=== Runtime Summary ({n_iter} iterations) ===")
print(f"Total   : {times_total.mean():.6f} ± {times_total.std():.6f} sec")
print(f"Encode  : {times_encode.mean():.6f} ± {times_encode.std():.6f} sec")
print(f"Project : {times_project.mean():.6f} ± {times_project.std():.6f} sec")

print(f"\n=== Memory Summary ({n_iter} iterations, peak MB) ===")
print(f"Total   : {mem_total.mean():.2f} ± {mem_total.std():.2f} MB")
print(f"Encode  : {mem_encode.mean():.2f} ± {mem_encode.std():.2f} MB")
print(f"Project : {mem_project.mean():.2f} ± {mem_project.std():.2f} MB")
print("Results limited to only CPU 1 core.")
print()

# CPU info
import platform
import cpuinfo

print("CPU threads PyTorch will use:", torch.get_num_threads())
print("Available devices:", torch.get_num_interop_threads())  # inter-op parallelism

print("Platform:", platform.system(), platform.release())
print("Machine:", platform.machine())

info = cpuinfo.get_cpu_info()
print("CPU brand:", info["brand_raw"])
print("Arch:", info["arch"])
print("Cores (logical):", info["count"])
print("Frequency (Hz):", info["hz_advertised_friendly"])


=== Runtime Summary (10 iterations) ===
Total   : 3.590838 ± 0.008699 sec
Encode  : 3.568635 ± 0.007730 sec
Project : 0.022203 ± 0.003829 sec

=== Memory Summary (10 iterations, peak MB) ===
Total   : 1394.39 ± 0.06 MB
Encode  : 1394.39 ± 0.06 MB
Project : 1202.53 ± 0.00 MB
Results limited to only CPU 1 core.

CPU threads PyTorch will use: 1
Available devices: 64
Platform: Linux 4.18.0-553.58.1.el8_10.x86_64
Machine: x86_64
CPU brand: Intel(R) Xeon(R) Gold 6338 CPU @ 2.00GHz
Arch: X86_64
Cores (logical): 64
Frequency (Hz): 2.0000 GHz


In [11]:
from tabulate import tabulate
import numpy as np

time_vae_load = time_after - time_before
time_cl_load = time_after_cl - time_after
time_total_load = time_after_cl - time_before

# Build table
summary = [
    ["Load VAE encoder", f"{time_vae_load:.2f}"],
    ["Load CL projector", f"{time_cl_load:.2f}"],
    ["Total load", f"{time_total_load:.2f}"],
    ["Run VAE Encode", f"{times_encode.mean():.2f} ± {times_encode.std():.2f}"],
    ["Run CL Project", f"{times_project.mean():.2f} ± {times_project.std():.2f}"],
    ["Total run", f"{times_total.mean():.2f} ± {times_total.std():.2f}"]
]

# Print table
print(tabulate(summary, headers=["Step", "Time (s)"], tablefmt="grid"))




+-------------------+-------------+
| Step              | Time (s)    |
+===================+=============+
| Load VAE encoder  | 0.55        |
+-------------------+-------------+
| Load CL projector | 0.55        |
+-------------------+-------------+
| Total load        | 1.10        |
+-------------------+-------------+
| Run VAE Encode    | 3.57 ± 0.01 |
+-------------------+-------------+
| Run CL Project    | 0.02 ± 0.00 |
+-------------------+-------------+
| Total run         | 3.59 ± 0.01 |
+-------------------+-------------+


**Results: 1 Core**
=== Runtime Summary (10 iterations) ===

Total   : 3.590838 ± 0.008699 sec

Encode  : 3.568635 ± 0.007730 sec

Project : 0.022203 ± 0.003829 sec

=== Memory Summary (10 iterations, peak MB) ===

Total   : 1394.39 ± 0.06 MB

Encode  : 1394.39 ± 0.06 MB

Project : 1202.53 ± 0.00 MB

Results limited to only CPU 1 core.

CPU threads PyTorch will use: 1

Available devices: 64

Platform: Linux 4.18.0-553.58.1.el8_10.x86_64

Machine: x86_64

CPU brand: Intel(R) Xeon(R) Gold 6338 CPU @ 2.00GHz

Arch: X86_64

Cores (logical): 64

Frequency (Hz): 2.0000 GHz

+-------------------+-------------+
| Step              | Time (s)    |


| Load VAE encoder  | 0.55        |

| Load CL projector | 0.55        |

| Total load        | 1.10        |

| Run VAE Encode    | 3.57 ± 0.01 |


| Run CL Project    | 0.02 ± 0.00 |


| Total run         | 3.59 ± 0.01 |
+-------------------+-------------+

**Results: 4 Core**

=== Runtime Summary (10 iterations) ===

Total   : 1.189344 ± 0.041044 sec

Encode  : 1.176916 ± 0.039271 sec

Project : 0.012427 ± 0.004687 sec

=== Memory Summary (10 iterations, peak MB) ===

Total   : 1492.20 ± 0.07 MB

Encode  : 1492.20 ± 0.07 MB

Project : 1236.33 ± 0.00 MB

Results limited to only CPU 1 core.

CPU threads PyTorch will use: 4

Available devices: 64

Platform: Linux 4.18.0-553.58.1.el8_10.x86_64

Machine: x86_64

CPU brand: Intel(R) Xeon(R) Gold 6338 CPU @ 2.00GHz

Arch: X86_64

Cores (logical): 64

Frequency (Hz): 2.0000 GHz


+-------------------+-------------+
| Step              | Time (s)    |

| Load VAE encoder  | 3.20        |

| Load CL projector | 2.08        |

| Total load        | 5.28        |

| Run VAE Encode    | 1.18 ± 0.04 |

| Run CL Project    | 0.01 ± 0.00 |

| Total run         | 1.19 ± 0.04 |
+-------------------+-------------+



In [12]:
import pandas as pd
import os

DATA_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/"

# Load and filter metadata
clinical_ds = pd.read_csv(os.path.join(DATA_PATH,"combined_metadata.csv"))

# Load real features from parquet
emb_path = os.path.join("data/results/region_brain/eval_cl32/", "projected_embeddings.parquet")  # e.g., "outputs/embeddings.parquet"
df_embs = pd.read_parquet(emb_path)

# Ensure GUID is string and joinable
df_embs["GUID"] = df_embs["GUID"].astype(str)
df_embs["LabelName"] = df_embs["LabelName"].astype(str)
clinical_ds["GUID"] = clinical_ds["GUID"].astype(str)

# Merge on GUID
dataset = pd.merge(clinical_ds, df_embs, on="GUID", how="inner")

# Convert embedding columns into a single 'features' column of vectors
embedding_cols = [col for col in df_embs.columns if not col in  ["GUID", "LabelName"]]
dataset["features"] = dataset[embedding_cols].apply(lambda row: row.to_numpy(), axis=1)

struct_name = "Left-Cerebral-White-Matter"
subset = dataset.query(f"LabelName == '{struct_name}'").reset_index(drop=True)
top_k_max = 5


In [13]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_topk_guid_retrievals(dataset: pd.DataFrame, top_k: int = 3, feature_column: str = 'features', guid_column: str = 'GUID') -> pd.DataFrame:
    """
    For each row in the dataset, retrieve the top-k most similar (cosine) entries and return their GUIDs.

    Args:
        dataset (pd.DataFrame): DataFrame with features and GUIDs.
        top_k (int): Number of top similar entries to retrieve.
        feature_column (str): Name of the column containing the feature vectors.
        guid_column (str): Column name for unique scan identifiers (e.g., 'GUID').

    Returns:
        pd.DataFrame: DataFrame with one row per query, first column is the query GUID,
                      and columns 1...k are the GUIDs of the top-k retrieved entries.
    """
    features_matrix = np.stack(dataset[feature_column].values)
    guids = dataset[guid_column].values
    retrievals = []
    times = []

    for i in tqdm(range(len(dataset))):
        start = time.perf_counter()

        similarities = cosine_similarity(features_matrix[i].reshape(1, -1), features_matrix)[0]
        similarities[i] = -1  # Exclude self
        top_k_indices = np.argsort(similarities)[::-1][:top_k]
        row = [guids[i]] + guids[top_k_indices].tolist()
        retrievals.append(row)

        end = time.perf_counter()
        times.append(end - start)

    # Compute statistics
    mean_time = np.mean(times)
    std_time = np.std(times)

    print(f"Mean time per iteration: {mean_time:.6f} s")
    print(f"Std of time per iteration: {std_time:.6f} s")

    col_names = ['query'] + [f'top{i+1}' for i in range(top_k)]
    return pd.DataFrame(retrievals, columns=col_names)

retrieval_df = get_topk_guid_retrievals(subset, top_k=top_k_max)

100%|██████████| 26198/26198 [01:04<00:00, 407.78it/s]

Mean time per iteration: 0.002431 s
Std of time per iteration: 0.000171 s


**Results: 1 core:**

100%|██████████| 26198/26198 [01:07<00:00, 388.51it/s]

Mean time per iteration: 0.002551 s

Std of time per iteration: 0.000404 s

**Results: 4 core:**

100%|██████████| 26198/26198 [01:11<00:00, 367.75it/s]

Mean time per iteration: 0.002694 s

Std of time per iteration: 0.000473 s

In [14]:
halt

NameError: name 'halt' is not defined

# WHOLE BRAIN

In [1]:
%cd /cephyr/users/felixnie/Alvis/NeuroCBIR/

/cephyr/users/felixnie/Alvis/NeuroCBIR


In [2]:
import matplotlib.pyplot as plt
import torch
import os
import psutil
import time
import gc
from tqdm import tqdm

gc.collect()  # Clean up any unreachable objects
%load_ext memory_profiler


# # Limit PyTorch to 1 thread
# torch.set_num_threads(1)

# # Optional: also restrict OMP and MKL
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"

# AUTOENCODER
device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
if "dataset" in locals():
    del dataset

from preprocessing.load_dataset import LookupNPZDataset
import pandas as pd
import os
import numpy as np

### Input data
# Path to dataset
DATA_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/"
load_ds_path = DATA_PATH + "batched_adni/"
# Files to load/save extension
extension = ".npz"
# Pretrained weights for the VAE
ckpt_vae_path = ""#"./data/pretrained_models/autoencoder_puglisi.pth"
# Pretrained weights for the Discriminator
ckpt_dis_path = ""#"./data/pretrained_models/ckpt_dis_.pth"
# Preparing image for using as input of the VAE
target_shape = [1, 160, 224, 160] # Desired shape: [1, 160, 224, 160]

# Load metadata
index_ds = pd.read_csv(os.path.join(DATA_PATH,"original/dataset_index.csv"))
clinical_ds = pd.read_csv(os.path.join(DATA_PATH,"combined_metadata.csv"))
metadata = pd.merge(index_ds, clinical_ds, on="GUID", how="inner") # Merge on the 'GUID' column
print(f"METADATA: Original rows: {len(metadata)}")

# First, ensure empty strings are treated as NaN
metadata['subject'].replace('', pd.NA, inplace=True)

# Then drop rows where subject is NaN
metadata = metadata.dropna(subset=['subject'])

# Optional: reset index
metadata = metadata.reset_index(drop=True)
print(f"METADATA: Remaining rows: {len(metadata)}") # Check result

# Load labels

# Training configuration
batch_files = sorted(metadata["batch_file"].unique())
batch_files = batch_files[-4:-3] # To only load ABLI
batch_file = batch_files[0]
print(batch_files)

from preprocessing.load_dataset import SingleStructDataset
# dataset = LookupNPZDataset(metadata, batch_file=batch_file, use_segmentation=False)
data = np.load(batch_file)

METADATA: Original rows: 26685
METADATA: Remaining rows: 26685
['/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/batched_aibl/batch_0002.npz']


In [4]:
config = {
    "seed": 42,
    "data_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/original/",
    "save_path": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/whole_brain/",
    "extension": ".npz",
    "metadata_file": "/mimer/NOBACKUP/groups/naiss2025-23-412/felixnie/batched_datasets/combined_metadata.csv",
    "ckpt_path": "/cephyr/users/felixnie/Alvis/logs/20250703_094042/checkpoint-epoch-8.pth",
    "ckpt_key": "autoencoder_state_dict",
    "use_old_state_dict": False,
    "vae_config": {
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 1,
        "channels": [
            64,
            128,
            128,
            128
        ],
        "latent_channels": 8,
        "num_res_blocks": 2,
        "norm_num_groups": 32,
        "norm_eps": 1e-6,
        "attention_levels": [
            False,
            False,
            False,
            False
        ],
        "with_encoder_nonlocal_attn": False,
        "with_decoder_nonlocal_attn": False,
        "include_fc": False
    }
}

In [5]:
from monai.networks.nets.autoencoderkl import AutoencoderKL
from memory_profiler import memory_usage

def load_vae_model(config, device, ckpt_path = None, use_old_state_dict = False):
    vae = AutoencoderKL(
        spatial_dims=config["spatial_dims"],
        in_channels=config["in_channels"],
        out_channels=config["out_channels"],
        channels=tuple(config["channels"]),
        latent_channels=config["latent_channels"],
        num_res_blocks=config["num_res_blocks"],
        norm_num_groups=config["norm_num_groups"],
        norm_eps=config["norm_eps"],
        attention_levels=tuple(config["attention_levels"]),
        with_encoder_nonlocal_attn=config["with_encoder_nonlocal_attn"],
        with_decoder_nonlocal_attn=config["with_decoder_nonlocal_attn"],
        include_fc=config["include_fc"]
    ).to(device)

    # Load weights (if checkpoint is provided)
    if ckpt_path:
        # Load full checkpoint
        checkpoint = torch.load(ckpt_path, map_location=device)
        ckpt_key = config.get("ckpt_key", "autoencoder_state_dict")
        # Load only the autoencoder weights
        if use_old_state_dict:
            vae.load_old_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_old_state_dict().")
        else:
            vae.load_state_dict(checkpoint[ckpt_key])
            print("Loaded weights using load_state_dict().")

    return vae

# -------------------------
# Baseline RAM before function call
# -------------------------
baseline_mem = memory_usage()[0]
print(f"RAM before process_batch: {baseline_mem:.2f} MB")


# Load VAE
autoencoder = load_vae_model(config=config["vae_config"],
                            device=device,
                            ckpt_path=config.get("ckpt_path"),
                            use_old_state_dict=config.get("use_old_state_dict")
                            )
autoencoder.eval()



RAM before process_batch: 858.24 MB
Loaded weights using load_state_dict().


AutoencoderKL(
  (encoder): Encoder(
    (blocks): ModuleList(
      (0): Convolution(
        (conv): Conv3d(1, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      )
      (1-2): 2 x AEKLResBlock(
        (norm1): GroupNorm(32, 64, eps=1e-06, affine=True)
        (conv1): Convolution(
          (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        )
        (norm2): GroupNorm(32, 64, eps=1e-06, affine=True)
        (conv2): Convolution(
          (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        )
        (nin_shortcut): Identity()
      )
      (3): AEKLDownsample(
        (pad): AsymmetricPad()
        (conv): Convolution(
          (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2))
        )
      )
      (4): AEKLResBlock(
        (norm1): GroupNorm(32, 64, eps=1e-06, affine=True)
        (conv1): Convolution(
          (conv): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=

In [6]:
dataset = {}
dataset['images'] = data['images'][0:5].copy()
dataset['GUID'] = data['GUID'][0:5].copy()

In [7]:
import time
import numpy as np
import torch
from tqdm import tqdm
from memory_profiler import memory_usage

def process_batch(data, autoencoder, device):
    times = []
    mem_usages = []

    z_mus = []
    ids = []

    images = data['images']
    sample_ids = data['GUID']

    for img, sample_id in tqdm(zip(images, sample_ids), total=len(images), ncols=150, desc="Processing batch"):
        
        img = np.expand_dims(img, axis=0).astype(np.float32) / 255.0
        img = torch.tensor(img).unsqueeze(0).to(device)

        # measure memory usage during the encode step
        def encode_fn():
            with torch.no_grad():
                start = time.perf_counter()
                z_mu, _ = autoencoder.encode(img)
                end = time.perf_counter()
            return z_mu, start, end

        mem, result = memory_usage(encode_fn, retval=True, max_usage=True)
        z_mu, start, end = result

        mem_usages.append(mem)  # peak RAM during this iteration (in MB)

        z_mu = z_mu.squeeze(0).cpu().numpy().astype(np.float16)
        z_mus.append(z_mu)
        ids.append(sample_id)

        times.append(end - start)

    mean_time = np.mean(times)
    std_time = np.std(times)
    mean_mem = np.mean(mem_usages)
    peak_mem = max(mem_usages)

    print(f"Mean time per iteration: {mean_time:.6f} s")
    print(f"Std of time per iteration: {std_time:.6f} s")
    print(f"Mean RAM usage per iteration: {mean_mem:.2f} MB")
    print(f"Peak RAM usage per iteration: {peak_mem:.2f} MB")

    return np.stack(z_mus), np.stack(ids)


# -------------------------
# Baseline RAM before function call
# -------------------------
baseline_mem = memory_usage()[0]
print(f"RAM before process_batch: {baseline_mem:.2f} MB")

# Run the function
z_mus, ids = process_batch(dataset, autoencoder, device)

# RAM after function
final_mem = memory_usage()[0]
print(f"RAM after process_batch: {final_mem:.2f} MB")


RAM before process_batch: 1087.61 MB


Processing batch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:34<00:00, 18.81s/it]

Mean time per iteration: 18.761493 s
Std of time per iteration: 0.460590 s
Mean RAM usage per iteration: 6766.37 MB
Peak RAM usage per iteration: 6771.52 MB
RAM after process_batch: 1006.88 MB


**Results: 1 core**

RAM before process_batch: 1072.08 MB

Processing batch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [05:32<00:00, 66.42s/it]

Mean time per iteration: 66.372468 s

Std of time per iteration: 0.551612 s

Mean RAM usage per iteration: 6800.16 MB

Peak RAM usage per iteration: 6804.56 MB

RAM after process_batch: 1084.59 MB

**Results: 4 cores**

RAM before process_batch: 1053.39 MB

Processing batch: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:26<00:00, 17.39s/it]

Mean time per iteration: 17.344226 s

Std of time per iteration: 0.122105 s

Mean RAM usage per iteration: 6764.95 MB

Peak RAM usage per iteration: 6765.37 MB

RAM after process_batch: 1045.26 MB


In [ ]:
device

'cpu'